# Business Entity Resolution — AWS Version (32GB RAM)

You already normalized everything on Colab and saved parquet checkpoints (`train_s1/s2/s3.parquet`, `ground_truth.parquet`, `test_s1/s2/s3.parquet`) to Google Drive. This notebook **fetches those parquet files directly** — no re-normalization — and runs a **wider, higher-recall blocking strategy plus richer features**, since 32GB RAM removes the memory ceiling that forced the tight Colab settings.

### What's different from the Colab version
- Blocking uses the full multi-token inverted-index approach (not the narrow single block-key), with a much looser `max_doc_freq`, because there's room for it now.
- Feature set is richer: adds TF-IDF cosine similarity on name+address (sklearn), on top of the fuzzy-match features.
- No mandatory restart-between-stages ritual — 32GB is enough headroom that this pipeline can run top to bottom in one session. Checkpointing is still kept so nothing is lost on any failure.

Run this on an AWS **SageMaker Notebook instance** (e.g. `ml.m5.2xlarge` = 32GB RAM, 8 vCPU) or a plain **EC2 instance** with Jupyter installed — either works, the code below is not SageMaker-specific.

## 0. Fetch your parquet checkpoints from Google Drive

**One-time step on Drive first:** right-click your `work` folder (the one containing all the `.parquet` files) → Share → "Anyone with the link" → Viewer. Copy the folder's link and grab the folder ID from the URL (the part after `folders/`).

Then run this cell on the AWS instance — it downloads the whole folder locally.

In [ ]:
!pip install -q gdown pandas pyarrow rapidfuzz lightgbm scikit-learn joblib


In [ ]:
import os

# ---- EDIT THIS: paste your Drive folder ID (from the shareable link) ----
DRIVE_FOLDER_ID = "PASTE_YOUR_FOLDER_ID_HERE"
# ---------------------------------------------------------------------

WORK_DIR = os.path.expanduser("~/entity_resolution_work")
os.makedirs(WORK_DIR, exist_ok=True)

def wpath(name):
    return os.path.join(WORK_DIR, name)

print("WORK_DIR:", WORK_DIR)


In [ ]:
import gdown

# Downloads every file from the shared Drive folder into WORK_DIR
gdown.download_folder(id=DRIVE_FOLDER_ID, output=WORK_DIR, quiet=False, use_cookies=False)

print("\nDownloaded files:")
for f in sorted(os.listdir(WORK_DIR)):
    print(" -", f)


**If `gdown.download_folder` fails** (Drive sometimes blocks folder-level downloads for large folders), fall back to downloading files one at a time. For each file, open it in Drive, right-click → Share → copy link → extract the file ID (the part after `/d/` and before `/view`), then:

```python
import gdown
gdown.download(id="FILE_ID_HERE", output=wpath("train_s1.parquet"), quiet=False)
# repeat for train_s2, train_s3, ground_truth, test_s1, test_s2, test_s3
```

**Alternative (often more reliable for many files / large folders):** zip the whole `work` folder in Drive first, share that single zip's link, download the zip with `gdown`, then `!unzip -q work.zip -d ~/entity_resolution_work`.

In [ ]:
# Verify everything needed is present before continuing
required = ["train_s1.parquet", "train_s2.parquet", "train_s3.parquet", "ground_truth.parquet",
            "test_s1.parquet", "test_s2.parquet", "test_s3.parquet"]
missing = [f for f in required if not os.path.exists(wpath(f))]
if missing:
    print("MISSING FILES — fetch these before continuing:", missing)
else:
    print("All required parquet files present.")


## 1. Imports and core functions

In [ ]:
import re, gc, unicodedata
from collections import defaultdict

import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)

def tokenset(t):
    return set(t.split()) if t else set()


### Load the normalized sources (already processed on Colab — just reading parquet, instant)

In [ ]:
s1 = pd.read_parquet(wpath("train_s1.parquet"))
s2 = pd.read_parquet(wpath("train_s2.parquet"))
s3 = pd.read_parquet(wpath("train_s3.parquet"))
ground_truth = pd.read_parquet(wpath("ground_truth.parquet"))

print("s1:", s1.shape, " s2:", s2.shape, " s3:", s3.shape, " ground_truth:", ground_truth.shape)
s1.head()


## 2. Blocking — wide multi-token inverted index (affordable now with 32GB RAM)

This is the full recall-focused version: every name token (down to `min_len=3`) and every address number is indexed, with a loose `max_doc_freq` so fewer useful tokens get dropped. This is exactly what caused OOM on 12GB — on 32GB it should comfortably fit, especially chunked.

In [ ]:
def build_other_token_index(other_df, text_col, min_len=3, max_doc_freq=0.02):
    tok = other_df[["entity_id"]].rename(columns={"entity_id": "other_entity_id"})
    tok["token"] = other_df[text_col].str.split()
    tok = tok.explode("token").dropna(subset=["token"])
    if min_len > 1:
        tok = tok[tok["token"].str.len() >= min_len]
    tok = tok.drop_duplicates()

    max_count = max(int(len(other_df) * max_doc_freq), 50)
    doc_freq = tok["token"].value_counts()
    keep = doc_freq[doc_freq <= max_count].index
    tok = tok[tok["token"].isin(keep)]

    tok["token"] = tok["token"].astype("category")
    tok["other_entity_id"] = tok["other_entity_id"].astype("category")
    return tok


def generate_candidates(s1_df, s2_df, s3_df, out_name,
                         min_len=3, max_doc_freq=0.02, s1_chunk_size=300000):
    out_path = wpath(f"{out_name}.parquet")
    if os.path.exists(out_path):
        print(f"[SKIP] {out_name} already generated -> {out_path}")
        return pd.read_parquet(out_path)

    print("Building S2/S3 token indexes once...")
    name_idx_s2 = build_other_token_index(s2_df, "name_core", min_len, max_doc_freq)
    name_idx_s3 = build_other_token_index(s3_df, "name_core", min_len, max_doc_freq)
    num_idx_s2  = build_other_token_index(s2_df, "addr_numbers", 1, max_doc_freq)
    num_idx_s3  = build_other_token_index(s3_df, "addr_numbers", 1, max_doc_freq)
    print("Indexes built:", len(name_idx_s2), len(name_idx_s3), len(num_idx_s2), len(num_idx_s3))

    results = []
    n = len(s1_df)

    for start in range(0, n, s1_chunk_size):
        s1_chunk = s1_df.iloc[start:start + s1_chunk_size]

        def s1_tokens(text_col, min_len):
            t = s1_chunk[["entity_id"]].rename(columns={"entity_id": "source1_entity_id"})
            t["token"] = s1_chunk[text_col].str.split()
            t = t.explode("token").dropna(subset=["token"])
            if min_len > 1:
                t = t[t["token"].str.len() >= min_len]
            return t.drop_duplicates()

        s1_name_tok = s1_tokens("name_core", min_len)
        s1_num_tok  = s1_tokens("addr_numbers", 1)

        p1 = s1_name_tok.merge(name_idx_s2, on="token", how="inner")[["source1_entity_id", "other_entity_id"]]
        p2 = s1_name_tok.merge(name_idx_s3, on="token", how="inner")[["source1_entity_id", "other_entity_id"]]
        p3 = s1_num_tok.merge(num_idx_s2, on="token", how="inner")[["source1_entity_id", "other_entity_id"]]
        p4 = s1_num_tok.merge(num_idx_s3, on="token", how="inner")[["source1_entity_id", "other_entity_id"]]

        all_pairs = pd.concat([p1, p2, p3, p4], ignore_index=True).drop_duplicates()
        del p1, p2, p3, p4, s1_name_tok, s1_num_tok

        grouped = all_pairs.astype(str).groupby("source1_entity_id")["other_entity_id"].apply(
            lambda x: ",".join(sorted(set(x)))
        ).reset_index()
        grouped.columns = ["source1_entity_id", "candidate_entity_ids"]
        results.append(grouped)

        del all_pairs, grouped
        gc.collect()
        print(f"  {min(start + s1_chunk_size, n):,}/{n:,} S1 rows done...")

    out = pd.concat(results, ignore_index=True)
    del results
    found_ids = set(out["source1_entity_id"])
    missing = [eid for eid in s1_df["entity_id"] if eid not in found_ids]
    if missing:
        out = pd.concat([out, pd.DataFrame({
            "source1_entity_id": missing, "candidate_entity_ids": [""] * len(missing)
        })], ignore_index=True)

    out.to_parquet(out_path, index=False)
    print(f"[SAVE] {out_name} -> {out_path}  ({len(out):,} rows)")
    return out


In [ ]:
train_candidates = generate_candidates(s1, s2, s3, "train_candidates_aws")
train_candidates.head()


### Check blocking recall

In [ ]:
cand = train_candidates.set_index("source1_entity_id")["candidate_entity_ids"]
gt = ground_truth.set_index("source1_entity_id")["matched_entity_ids"]

total_true, found_true = 0, 0
for s1_id, true_ids_str in gt.items():
    true_ids = set(x for x in true_ids_str.split(",") if x)
    if not true_ids:
        continue
    cand_ids = set(cand.get(s1_id, "").split(",")) if s1_id in cand.index else set()
    total_true += len(true_ids)
    found_true += len(true_ids & cand_ids)

recall = found_true / total_true if total_true else float("nan")
print(f"Blocking recall ceiling: {recall:.4f}  ({found_true}/{total_true})")


If recall is still not high enough, loosen further: lower `min_len` to 2, raise `max_doc_freq` to 0.03-0.05, or lower `s1_chunk_size` if that causes memory pressure. With 32GB there's real room to push this.

## 3. Feature engineering — richer feature set (TF-IDF cosine added, affordable with more compute)

In [ ]:
def expand_pairs(candidates_df):
    rows = []
    for s1_id, cand_str in zip(candidates_df["source1_entity_id"], candidates_df["candidate_entity_ids"]):
        if not cand_str:
            continue
        for cid in cand_str.split(","):
            rows.append((s1_id, cid))
    return pd.DataFrame(rows, columns=["source1_entity_id", "candidate_entity_id"])

train_pairs = expand_pairs(train_candidates)
print(train_pairs.shape)


In [ ]:
def build_feature_table(pairs_df, s1_df, s2_df, s3_df, out_name, batch_size=200000):
    out_path = wpath(f"{out_name}.parquet")
    if os.path.exists(out_path):
        print(f"[SKIP] {out_name} already built -> {out_path}")
        return pd.read_parquet(out_path)

    s1_idx = s1_df.set_index("entity_id")[["name_core", "address_norm", "country"]]
    other_idx = pd.concat([s2_df, s3_df])[["entity_id", "name_core", "address_norm", "country"]].set_index("entity_id")

    # fit one shared TF-IDF vectorizer on name+address text across all sources (char n-grams handle typos well)
    print("Fitting TF-IDF vectorizer...")
    all_text = pd.concat([
        (s1_df["name_core"] + " " + s1_df["address_norm"]),
        (s2_df["name_core"] + " " + s2_df["address_norm"]),
        (s3_df["name_core"] + " " + s3_df["address_norm"]),
    ])
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=50000)
    vectorizer.fit(all_text)
    del all_text
    gc.collect()

    s1_text_map = (s1_df["name_core"] + " " + s1_df["address_norm"]).set_axis(s1_df["entity_id"])
    other_text_map = pd.concat([
        (s2_df["name_core"] + " " + s2_df["address_norm"]).set_axis(s2_df["entity_id"]),
        (s3_df["name_core"] + " " + s3_df["address_norm"]).set_axis(s3_df["entity_id"]),
    ])

    def jaccard(a, b):
        sa, sb = tokenset(a), tokenset(b)
        if not sa and not sb:
            return 1.0
        u = sa | sb
        return len(sa & sb) / len(u) if u else 0.0

    n = len(pairs_df)
    frames = []

    for start in range(0, n, batch_size):
        chunk = pairs_df.iloc[start:start + batch_size]

        name1 = chunk["source1_entity_id"].map(s1_idx["name_core"]).fillna("")
        addr1 = chunk["source1_entity_id"].map(s1_idx["address_norm"]).fillna("")
        c1    = chunk["source1_entity_id"].map(s1_idx["country"]).astype(str)

        name2 = chunk["candidate_entity_id"].map(other_idx["name_core"]).fillna("")
        addr2 = chunk["candidate_entity_id"].map(other_idx["address_norm"]).fillna("")
        c2    = chunk["candidate_entity_id"].map(other_idx["country"]).astype(str)

        text1 = chunk["source1_entity_id"].map(s1_text_map).fillna("")
        text2 = chunk["candidate_entity_id"].map(other_text_map).fillna("")

        X1 = vectorizer.transform(text1)
        X2 = vectorizer.transform(text2)
        tfidf_cos = np.asarray(X1.multiply(X2).sum(axis=1)).ravel()
        norms1 = np.sqrt(np.asarray(X1.multiply(X1).sum(axis=1)).ravel())
        norms2 = np.sqrt(np.asarray(X2.multiply(X2).sum(axis=1)).ravel())
        denom = norms1 * norms2
        tfidf_cos = np.divide(tfidf_cos, denom, out=np.zeros_like(tfidf_cos), where=denom != 0)

        feats = pd.DataFrame({
            "source1_entity_id": chunk["source1_entity_id"].values,
            "candidate_entity_id": chunk["candidate_entity_id"].values,
            "name_ratio": np.array([fuzz.token_sort_ratio(a, b) for a, b in zip(name1, name2)], dtype=np.float32),
            "name_partial_ratio": np.array([fuzz.partial_ratio(a, b) for a, b in zip(name1, name2)], dtype=np.float32),
            "addr_ratio": np.array([fuzz.token_sort_ratio(a, b) for a, b in zip(addr1, addr2)], dtype=np.float32),
            "name_jaccard": np.array([jaccard(a, b) for a, b in zip(name1, name2)], dtype=np.float32),
            "addr_jaccard": np.array([jaccard(a, b) for a, b in zip(addr1, addr2)], dtype=np.float32),
            "tfidf_cosine": tfidf_cos.astype(np.float32),
            "same_country": (c1.values == c2.values).astype(np.int8),
            "name_len_diff": np.abs(name1.str.len().values - name2.str.len().values).astype(np.float32),
        })
        frames.append(feats)
        print(f"  processed {min(start + batch_size, n):,}/{n:,} pairs")

    out = pd.concat(frames, ignore_index=True)
    out.to_parquet(out_path, index=False)
    print(f"[SAVE] {out_name} -> {out_path}")
    return out

FEATURE_COLS = ["name_ratio", "name_partial_ratio", "addr_ratio",
                 "name_jaccard", "addr_jaccard", "tfidf_cosine", "same_country", "name_len_diff"]


In [ ]:
train_features = build_feature_table(train_pairs, s1, s2, s3, "train_features_aws")
train_features.head()


## 4. Labels + train/validation split

In [ ]:
def build_labels(pairs_df, ground_truth_df):
    gt_map = {}
    for s1_id, matched in zip(ground_truth_df["source1_entity_id"], ground_truth_df["matched_entity_ids"]):
        gt_map[s1_id] = set(x for x in matched.split(",") if x)
    labels = [
        1 if cid in gt_map.get(s1_id, set()) else 0
        for s1_id, cid in zip(pairs_df["source1_entity_id"], pairs_df["candidate_entity_id"])
    ]
    return pd.Series(labels, name="label", dtype=np.int8)

train_features["label"] = build_labels(train_features, ground_truth)
print(train_features["label"].value_counts())


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(train_features, groups=train_features["source1_entity_id"]))

X_train = train_features.iloc[train_idx][FEATURE_COLS]
y_train = train_features.iloc[train_idx]["label"]
X_val   = train_features.iloc[val_idx][FEATURE_COLS]
y_val   = train_features.iloc[val_idx]["label"]

print("Train pairs:", X_train.shape, " Val pairs:", X_val.shape)


## 5. Train LightGBM (more trees now — compute headroom allows a stronger model) + tune threshold for F0.5

In [ ]:
import lightgbm as lgb
import joblib

model = lgb.LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)
joblib.dump(model, wpath("model_aws.pkl"))

val_proba = model.predict_proba(X_val)[:, 1]


In [ ]:
def f_beta(precision, recall, beta=0.5):
    if precision == 0 and recall == 0:
        return 0.0
    b2 = beta ** 2
    return (1 + b2) * precision * recall / (b2 * precision + recall + 1e-12)

def macro_f05_per_entity(pairs_df, proba, threshold, ground_truth_df, s1_ids):
    pred_map = defaultdict(set)
    for s1_id, cid, p in zip(pairs_df["source1_entity_id"], pairs_df["candidate_entity_id"], proba):
        if p >= threshold:
            pred_map[s1_id].add(cid)

    gt_map = {}
    for s1_id, matched in zip(ground_truth_df["source1_entity_id"], ground_truth_df["matched_entity_ids"]):
        gt_map[s1_id] = set(x for x in matched.split(",") if x)

    scores = []
    for s1_id in s1_ids:
        true_ids = gt_map.get(s1_id, set())
        pred_ids = pred_map.get(s1_id, set())
        if not true_ids and not pred_ids:
            scores.append(1.0); continue
        if not true_ids and pred_ids:
            scores.append(0.0); continue
        tp = len(true_ids & pred_ids)
        precision = tp / len(pred_ids) if pred_ids else 0.0
        recall = tp / len(true_ids) if true_ids else 0.0
        scores.append(f_beta(precision, recall))
    return float(np.mean(scores))

val_df = train_features.iloc[val_idx][["source1_entity_id", "candidate_entity_id"]]
val_s1_ids = val_df["source1_entity_id"].unique()

best_t, best_f = 0.5, -1
for t in np.arange(0.3, 0.96, 0.05):
    f = macro_f05_per_entity(val_df, val_proba, t, ground_truth, val_s1_ids)
    print(f"threshold={t:.2f}  macro F0.5={f:.4f}")
    if f > best_f:
        best_f, best_t = f, t

print(f"\nBest threshold: {best_t:.2f}   Best macro F0.5: {best_f:.4f}")
with open(wpath("best_threshold_aws.txt"), "w") as f:
    f.write(str(best_t))


## 6. Test set — same pipeline, reused functions

In [ ]:
test_s1 = pd.read_parquet(wpath("test_s1.parquet"))
test_s2 = pd.read_parquet(wpath("test_s2.parquet"))
test_s3 = pd.read_parquet(wpath("test_s3.parquet"))
print(test_s1.shape, test_s2.shape, test_s3.shape)


In [ ]:
test_candidates = generate_candidates(test_s1, test_s2, test_s3, "test_candidates_aws")
test_pairs = expand_pairs(test_candidates)
print(test_pairs.shape)


In [ ]:
test_features = build_feature_table(test_pairs, test_s1, test_s2, test_s3, "test_features_aws")

model = joblib.load(wpath("model_aws.pkl"))
with open(wpath("best_threshold_aws.txt")) as f:
    best_t = float(f.read().strip())

test_proba = model.predict_proba(test_features[FEATURE_COLS])[:, 1]
test_features["score"] = test_proba.astype(np.float32)
test_features["predicted"] = (test_proba >= best_t).astype(np.int8)
test_features.to_parquet(wpath("test_predictions_aws.parquet"), index=False)
print(test_features["predicted"].value_counts())


## 7. Write output files + validate locally

In [ ]:
OUTPUT_DIR = os.path.join(WORK_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
all_test_s1_ids = test_s1["entity_id"].tolist()

matches_map = defaultdict(list)
for s1_id, cid, pred in zip(test_features["source1_entity_id"], test_features["candidate_entity_id"], test_features["predicted"]):
    if pred == 1:
        matches_map[s1_id].append(cid)

match_rows = [(eid, ",".join(sorted(set(matches_map.get(eid, []))))) for eid in all_test_s1_ids]
matching_results = pd.DataFrame(match_rows, columns=["source1_entity_id", "matched_entity_ids"])
matching_results.to_csv(os.path.join(OUTPUT_DIR, "matching_results.tsv"), sep="\t", index=False)
print("Saved matching_results.tsv", matching_results.shape)


In [ ]:
cand_map = defaultdict(list)
for s1_id, cid in zip(test_features["source1_entity_id"], test_features["candidate_entity_id"]):
    cand_map[s1_id].append(cid)

cand_rows = [(eid, ",".join(sorted(set(cand_map.get(eid, []))))) for eid in all_test_s1_ids]
candidate_pairs_out = pd.DataFrame(cand_rows, columns=["source1_entity_id", "candidate_entity_ids"])
candidate_pairs_out.to_csv(os.path.join(OUTPUT_DIR, "candidate_pairs.tsv"), sep="\t", index=False)
print("Saved candidate_pairs.tsv", candidate_pairs_out.shape)


In [ ]:
valid_other_ids = set(test_s2["entity_id"]) | set(test_s3["entity_id"])
issues = []

if len(matching_results) != len(all_test_s1_ids):
    issues.append("Row count mismatch: not every S1 test entity has exactly one row.")
if matching_results["source1_entity_id"].duplicated().any():
    issues.append("Duplicate source1_entity_id rows in matching_results.")

cand_lookup = candidate_pairs_out.set_index("source1_entity_id")["candidate_entity_ids"]

for s1_id, ids_str in zip(matching_results["source1_entity_id"], matching_results["matched_entity_ids"]):
    ids = [x for x in ids_str.split(",") if x]
    if len(ids) != len(set(ids)):
        issues.append(f"Duplicate IDs within list for {s1_id}")
    bad = [i for i in ids if i not in valid_other_ids]
    if bad:
        issues.append(f"{s1_id} has IDs not in test S2/S3: {bad}")
    cand_ids = set(cand_lookup.get(s1_id, "").split(",")) if s1_id in cand_lookup.index else set()
    leftover = set(ids) - cand_ids
    if leftover:
        issues.append(f"{s1_id}: matched IDs not present in candidate_pairs -> {leftover}")

if issues:
    print(f"FOUND {len(issues)} ISSUE(S):")
    for i in issues[:30]:
        print(" -", i)
else:
    print("PASS: local checks look good. Also run the official utils/validate_submission.py before submitting.")


## Download outputs back to your machine for submission

If this is a SageMaker notebook instance, use the Jupyter file browser to download `output/matching_results.tsv` and `output/candidate_pairs.tsv` directly. On plain EC2, `scp` them:
```bash
scp -i your-key.pem ubuntu@<EC2_IP>:~/entity_resolution_work/output/*.tsv .
```